## 1. Loading & Cleaning the Data

In [1]:
import pandas as pd
import requests
import time
import os
from tqdm import tqdm

print("--- Phase 1: Loading & Cleaning ClinVar Data ---")
file_path = '/kaggle/input/datasets/competitiveracist/clinvar-tab-delimited/variant_summary.txt'

# Read the .txt file using tabs
df = pd.read_csv(file_path, sep='\t', low_memory=False)
df = df[df['Assembly'] == 'GRCh38']

# Drop missing VCF positions
df = df.dropna(subset=['ReferenceAlleleVCF', 'AlternateAlleleVCF', 'PositionVCF'])
df = df[df['PositionVCF'] != -1]

# Filter for SNPs only (Ensembl API breaks on massive indels)
df = df[
    (df['ReferenceAlleleVCF'].astype(str).str.len() == 1) &
    (df['AlternateAlleleVCF'].astype(str).str.len() == 1)
]

# Filter for clear Pathogenic or Benign cases (drops "Uncertain significance")
pathogenic_mask = df['ClinicalSignificance'].astype(str).str.contains('Pathogenic', case=False, na=False)
benign_mask = df['ClinicalSignificance'].astype(str).str.contains('Benign', case=False, na=False)
df = df[pathogenic_mask | benign_mask].copy()

# Create the clean binary target column
df['Target'] = df['ClinicalSignificance'].apply(lambda x: 1 if 'pathogenic' in str(x).lower() else 0)

print(f"Total clean SNP rows ready to process: {len(df)}")

--- Phase 1: Loading & Cleaning ClinVar Data ---
Total clean SNP rows ready to process: 1615872


In [2]:
df.head()

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,Target
7,15044,single nucleotide variant,NM_017547.4(FOXRED1):c.694C>T (p.Gln232Ter),55572,FOXRED1,HGNC:26927,Pathogenic,1,"Aug 17, 2025",267606829,...,-,-,-,-,-,-,SCV000680696|SCV001363290|SCV002793147|SCV0029...,-,-,1
9,15045,single nucleotide variant,NM_017547.4(FOXRED1):c.1289A>G (p.Asn430Ser),55572,FOXRED1,HGNC:26927,Likely pathogenic,1,"Jun 06, 2024",267606830,...,-,-,-,-,-,-,SCV005680614,-,-,1
11,15046,single nucleotide variant,NM_025152.3(NUBPL):c.166G>A (p.Gly56Arg),80224,NUBPL,HGNC:20278,Conflicting classifications of pathogenicity,1,"Apr 08, 2025",200401432,...,-,-,-,-,-,-,SCV000251971|SCV000742093|SCV001736868|SCV0032...,-,-,1
13,15048,single nucleotide variant,NM_000410.4(HFE):c.845G>A (p.Cys282Tyr),3077,HFE,HGNC:4886,Conflicting classifications of pathogenicity; ...,1,"Feb 03, 2026",1800562,...,-,-,-,-,-,-,SCV000151394|SCV000206975|SCV000219175|SCV0002...,-,-,1
15,15049,single nucleotide variant,NM_000410.4(HFE):c.187C>G (p.His63Asp),3077,HFE,HGNC:4886,Conflicting classifications of pathogenicity; ...,1,"Feb 03, 2026",1799945,...,-,-,-,-,-,-,SCV000206973|SCV000219176|SCV000223933|SCV0002...,-,-,1


## 2. The API Extraction Engine

In [3]:
# none

## 3. The Checkpoint Scraping Loop

In [4]:
import os
import requests
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

print("--- Phase 3: The Bulk-Batched Multi-Threaded Scraper ---")

# 1. THE GOLDEN RULE: Deduplication
# This instantly deletes hundreds of thousands of useless duplicate API pings
df_unique = df.drop_duplicates(subset=['Chromosome', 'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF']).copy()
print(f"Removed duplicates. Unique mutations left to scrape: {len(df_unique)}")

# 2. PREPARE THE BATCHES
# Ensembl's maximum allowed limit for a POST request is 200 variants
BATCH_SIZE = 200 
batches = [df_unique.iloc[i:i + BATCH_SIZE] for i in range(0, len(df_unique), BATCH_SIZE)]
print(f"Divided into {len(batches)} batches of {BATCH_SIZE} variants.")

output_file = '/kaggle/working/massive_enriched_dataset.csv'
if not os.path.isfile(output_file):
    with open(output_file, 'w') as f:
        f.write("Variant_ID,gnomAD_AF,SIFT_Score,CADD_Score,Is_Nonsense,Is_Missense,Is_Synonymous,Is_Splice_Site,Is_Pathogenic\n")

# Resume logic: Read what we already have so we don't double-scrape
try:
    existing_df = pd.read_csv(output_file)
    completed_variants = set(existing_df['Variant_ID'].values)
    print(f"Found {len(completed_variants)} already scraped variants. Resuming...")
except pd.errors.EmptyDataError:
    completed_variants = set()

# 3. THE BULK WORKER FUNCTION
def process_batch(batch_df):
    server = "https://rest.ensembl.org"
    # Using the POST endpoint instead of GET
    ext = "/vep/human/region?af=1&CADD=1"
    headers = { "Content-Type": "application/json", "Accept": "application/json"}

    payload_variants = []
    metadata = {} # To remember which target maps to which variant

    for _, row in batch_df.iterrows():
        chrom = str(row['Chromosome']).replace('chr', '')
        pos = int(row['PositionVCF'])
        ref = row['ReferenceAlleleVCF']
        alt = row['AlternateAlleleVCF']
        target = row['Target']
        var_id = f"{chrom}:{pos}:{ref}:{alt}"

        if var_id in completed_variants:
            continue # Skip if we already saved this one

        # The Ensembl bulk string format requires the strand (usually 1)
        query_str = f"{chrom}:{pos}-{pos}:1/{alt}"
        payload_variants.append(query_str)
        metadata[query_str] = {"var_id": var_id, "target": target, "alt": alt}

    # If the whole batch was already scraped, return empty list
    if not payload_variants:
        return [] 

    results = []
    try:
        # Send all 200 variants in ONE network ping
        res = requests.post(server + ext, headers=headers, json={"variants": payload_variants}, timeout=30)
        if res.ok:
            data_list = res.json()
            
            # Loop through the massive response payload
            for data in data_list:
                q_str = data.get('input')
                if not q_str or q_str not in metadata: continue

                meta = metadata[q_str]
                alt_allele = meta["alt"]

                # Extract features exactly like before
                af, sift, cadd = 0.0, 0.05, 0.0
                nonsense, missense, syn, splice = 0, 0, 0, 0

                for colocated in data.get('colocated_variants', []):
                    freqs = colocated.get('frequencies', {}).get(alt_allele, {})
                    for key in ['gnomad', 'gnomade', 'gnomadg']:
                        if key in freqs:
                            af = float(freqs[key])
                            break
                    if af > 0.0: break

                sift_temp, sift_found = 1.0, False
                for t in data.get('transcript_consequences', []):
                    terms = t.get('consequence_terms', [])
                    if 'stop_gained' in terms: nonsense = 1
                    if 'missense_variant' in terms: missense = 1
                    if 'synonymous_variant' in terms: syn = 1
                    if any('splice' in term for term in terms): splice = 1

                    if 'sift_score' in t:
                        sift_found = True
                        if t['sift_score'] < sift_temp: sift_temp = t['sift_score']
                    if 'cadd_phred' in t and t['cadd_phred'] > cadd:
                        cadd = t['cadd_phred']

                if sift_found: sift = sift_temp

                # Format the final CSV string
                results.append(f"{meta['var_id']},{af},{sift},{cadd},{nonsense},{missense},{syn},{splice},{meta['target']}\n")
    except Exception as e:
        # If Ensembl completely crashes on a batch, we just pass and retry later
        pass 

    return results

# 4. THE THREAD POOL MANAGER
MAX_THREADS = 10 
buffer = []

print(f"Igniting Bulk API Scraper with {MAX_THREADS} threads. Hold on tight...")

with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
    futures = [executor.submit(process_batch, batch) for batch in batches]
    
    # We use tqdm on the BATCHES, not individual rows.
    for future in tqdm(as_completed(futures), total=len(futures), desc="Bulk Scraping"):
        try:
            batch_results = future.result()
            buffer.extend(batch_results)
            
            # Checkpoint every ~1000 rows safely
            if len(buffer) >= 1000:
                with open(output_file, 'a') as f:
                    f.writelines(buffer)
                buffer = [] 
        except Exception:
            pass

if buffer:
    with open(output_file, 'a') as f:
        f.writelines(buffer)

print("🎉 BOOM! Bulk scraping complete. Dataset is fully loaded.")

--- Phase 3: The Bulk-Batched Multi-Threaded Scraper ---
Removed duplicates. Unique mutations left to scrape: 1615872
Divided into 8080 batches of 200 variants.
Found 0 already scraped variants. Resuming...
Igniting Bulk API Scraper with 10 threads. Hold on tight...


Bulk Scraping: 100%|██████████| 8080/8080 [5:47:18<00:00,  2.58s/it]

🎉 BOOM! Bulk scraping complete. Dataset is fully loaded.


In [5]:
import shutil

print("🗜️ Compressing the massive dataset for easy download...")
# shutil.make_archive automatically adds the .zip extension to your base_name
shutil.make_archive(
    base_name='/kaggle/working/massive_dataset', 
    format='zip', 
    root_dir='/kaggle/working', 
    base_dir='massive_enriched_dataset.csv'
)
print("Zipped successfully! You are clear for takeoff.")

🗜️ Compressing the massive dataset for easy download...
Zipped successfully! You are clear for takeoff.
